# Qwen2.5-7B-Instruct → FP8: quantize, check accuracy, profile

**Session 1 of 2 — llmcompressor / transformers** (Google Colab, NVIDIA L4 24 GB).

llmcompressor and vLLM 0.7.3 pin incompatible torch / transformers / compressed-tensors versions, so
quantization and evaluation run here and serving runs in `02_serve_and_benchmark.ipynb` in a fresh
runtime.

1. FP8_DYNAMIC quantization: FP8 weights with per-channel scales, per-token activation scales computed
   at runtime, so **no calibration data** is needed
2. Patch a transformers-5-only tokenizer field so the older serving stack can load the checkpoint
3. Perplexity on WikiText-2: BF16 vs FP8
4. Op-level profile of the plain-transformers load path

> Outputs are from the recorded run; install logs and progress bars were stripped.

## 0. Install, restart, verify

In [ ]:
!pip install llmcompressor -q
!pip install transformers -q          # let llmcompressor pull the version it wants
!pip install datasets -q
!pip uninstall torchvision -y
# import os; os.kill(os.getpid(), 9)

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'torchvision', '-y'], capture_output=True)

import torch, transformers
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor import oneshot
print(f"torch {torch.__version__} | transformers {transformers.__version__}")

torch 2.11.0+cu128 | transformers 5.17.0


In [ ]:
from google.colab import drive; drive.mount('/content/drive')

MODEL_ID     = "Qwen/Qwen2.5-7B-Instruct"
MODEL_FP8    = "/content/drive/MyDrive/qwen2.5-7b-fp8"

## 1. Quantize to FP8_DYNAMIC (`lm_head` stays BF16)

In [2]:

# Sanity check versions actually resolved before committing GPU time:
import llmcompressor, transformers, torch
print(f"llmcompressor {llmcompressor.__version__} | transformers {transformers.__version__} | torch {torch.__version__}")
# If this errors or versions look unexpected, stop here and re-check CELL B1
# output before spending premium GPU time on the quantization run below.

from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor import oneshot
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="bfloat16", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# FP8_DYNAMIC computes scales at inference time -> no calibration set needed.
# This is the key time-saver vs GPTQ, which needed 256 calibration passes.
recipe = QuantizationModifier(targets="Linear", scheme="FP8_DYNAMIC", ignore=["lm_head"])

oneshot(model=model, recipe=recipe, output_dir=MODEL_FP8)
tokenizer.save_pretrained(MODEL_FP8)

del model
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("FP8 quantization done ->", MODEL_FP8)


llmcompressor 0.14.0 | transformers 5.17.0 | torch 2.11.0+cu128


FP8 quantization done -> /content/drive/MyDrive/qwen2.5-7b-fp8


## 2. Make the checkpoint loadable by the serving stack
transformers 5.x writes `extra_special_tokens` as a list; transformers 4.x (used by vLLM 0.7.3) expects
a dict. The tokens stay registered via `special_tokens_map.json`, so emptying the field is safe.

In [3]:
import json

config_path = f"{MODEL_FP8}/tokenizer_config.json"
with open(config_path) as f:
    tok_config = json.load(f)

if isinstance(tok_config.get("extra_special_tokens"), list):
    n = len(tok_config["extra_special_tokens"])
    print(f"Patching extra_special_tokens: list ({n} tokens) -> empty dict")
    tok_config["extra_special_tokens"] = {}
    with open(config_path, "w") as f:
        json.dump(tok_config, f, indent=2)
    print("Patched. These tokens remain registered via special_tokens_map.json / "
          "added_tokens_decoder -- this only disables an optional v5-only convenience field.")
else:
    print("No patch needed -- already dict or field absent.")

Patching extra_special_tokens: list (13 tokens) -> empty dict
Patched. These tokens remain registered via special_tokens_map.json / added_tokens_decoder -- this only disables an optional v5-only convenience field.


## 3. Accuracy: WikiText-2 perplexity, BF16 vs FP8
Sliding window (1,024 tokens, stride 512), first 20 windows. Must run in **this** session: loading a
compressed checkpoint through transformers needs the newer compressed-tensors.

In [4]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

def compute_perplexity(model_path, stride=512, max_length=1024, n_samples=None):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, dtype="auto", device_map="auto")
    model.eval()

    test = test = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    text = "\\n\\n".join(test["text"])
    encodings = tokenizer(text, return_tensors="pt")
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end = 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = encodings.input_ids[:, begin:end].to(model.device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            out = model(input_ids, labels=target_ids)
            nlls.append(out.loss * trg_len)

        prev_end = end
        if end == seq_len:
            break
        if n_samples and len(nlls) >= n_samples:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / prev_end)
    del model
    import gc; gc.collect(); torch.cuda.empty_cache()
    return ppl.item()

ppl_baseline = compute_perplexity(MODEL_ID, n_samples=20)   # cap samples to save time/budget
ppl_fp8      = compute_perplexity(MODEL_FP8, n_samples=20)
print(f"Baseline PPL: {ppl_baseline:.3f} | FP8 PPL: {ppl_fp8:.3f} | Delta: {ppl_fp8 - ppl_baseline:+.3f}")

Baseline PPL: 6.469 | FP8 PPL: 6.536 | Delta: +0.066


**Result:** 6.469 → 6.536 (+1.0%), a small cost for halving weight memory.

## 4. Profile (read the caveat)
Plain transformers has no FP8 kernels: it **decompresses the FP8 weights to BF16** on load, so this
profile mostly shows dequantization (`aten::mul`, `aten::to`/`copy_` ≈ 75% of CUDA time) and BF16
GEMMs (`cutlass_80_wmma_tensorop_bf16`). It is *not* a measurement of FP8 inference; that is what the
vLLM benchmarks in notebook 02 measure. The GPU sampler at the end ran with no workload inside its
window, so its utilization/power line is not meaningful.

In [5]:
# --- (1) Op-level breakdown ---
import torch
from torch.profiler import profile, ProfilerActivity
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_FP8)
model = AutoModelForCausalLM.from_pretrained(MODEL_FP8, torch_dtype="auto", device_map="auto")
inputs = tokenizer("Explain FP8 quantization.", return_tensors="pt").to(model.device)

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    with torch.no_grad():
        model(**inputs)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))
prof.export_chrome_trace("trace.json")  # download and open at chrome://tracing

del model
import gc; gc.collect(); torch.cuda.empty_cache()

# --- (2) Background utilization/memory sampling ---
import subprocess, threading, time

gpu_log = []
stop_flag = threading.Event()

def sample_gpu(interval=0.5):
    while not stop_flag.is_set():
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True,
        )
        util, mem_used, mem_total, power = out.stdout.strip().split(", ")
        gpu_log.append((time.time(), int(util), int(mem_used), int(mem_total), float(power)))
        time.sleep(interval)

sampler = threading.Thread(target=sample_gpu)
sampler.start()

# --- run your benchmark here while sampling, e.g.: ---
# asyncio.run(run_load_test(50))

stop_flag.set()
sampler.join()

avg_util = sum(r[1] for r in gpu_log) / len(gpu_log)
peak_mem = max(r[2] for r in gpu_log)
avg_power = sum(r[4] for r in gpu_log) / len(gpu_log)
print(f"Avg GPU utilization: {avg_util:.1f}% | Peak memory: {peak_mem} MB | Avg power: {avg_power:.0f}W")
# Combine with your system_tps number from CELL G to get tokens/sec/watt (efficiency).

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              aten::mul         2.01%      10.400ms         4.64%      24.023ms      37.073us     102.372ms        39.15%     102.372ms     157.982us           0 B           0 B      12.22 GB      12.22 G